#  Sample Extraction

**Runs locally** — requires access to `cocotext.v2.json`.

Produces two JSON files:
- `04_train_sample.json` — 300 train images with GT boxes and text
- `04_test_sample.json`  — 200 val images with GT boxes and text

Upload both to `outputs/results/` on Drive, then run `04b_full_benchmark.ipynb` on Colab.

## 1. Paths & Config

In [1]:
import json, random
from pathlib import Path

# ── Update these to match your local paths ──
COCO_DIR    = Path('../data/raw')
ANN_FILE    = COCO_DIR / 'cocotext.v2.json'
OUTPUT_DIR  = Path('../outputs/results')
# ────────────────────────────────────────────

SEED    = 42
N_TRAIN = 2000
N_TEST  = 1000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert ANN_FILE.exists(), f'Annotation file not found: {ANN_FILE}'
print('Paths OK ')

Paths OK 


## 2. Load & Filter Annotations

In [2]:
print('Loading COCO-Text annotations...')
with open(ANN_FILE) as f:
    coco = json.load(f)

# Build image_id → annotations lookup
# Keep only: legible, English, text length >= 2
img_to_anns = {}
for ann_id, ann in coco['anns'].items():
    if ann.get('legibility') != 'legible': continue
    if ann.get('language')   != 'english': continue
    utf8 = ann.get('utf8_string', '').strip()
    if len(utf8) < 2: continue
    iid = ann['image_id']
    if iid not in img_to_anns:
        img_to_anns[iid] = []
    x, y, w, h = ann['bbox']
    img_to_anns[iid].append({
        'box':  [float(x), float(y), float(w), float(h)],
        'text': utf8
    })

# Normalize keys to int
img_to_anns_int = {int(k): v for k, v in img_to_anns.items()}
imgs_int        = {int(k): v for k, v in coco['imgs'].items()}

train_ids = [iid for iid, info in imgs_int.items()
             if info.get('set') == 'train' and iid in img_to_anns_int]
val_ids   = [iid for iid, info in imgs_int.items()
             if info.get('set') == 'val'   and iid in img_to_anns_int]

print(f'Eligible train images : {len(train_ids)}')
print(f'Eligible val   images : {len(val_ids)}')

Loading COCO-Text annotations...
Eligible train images : 12303
Eligible val   images : 2899


## 3. Sample & Save JSONs

In [3]:
random.seed(SEED)
train_sample = random.sample(train_ids, min(N_TRAIN, len(train_ids)))
test_sample  = random.sample(val_ids,   min(N_TEST,  len(val_ids)))

def build_sample(ids):
    return [{
        'image_id':  iid,
        'file_name': imgs_int[iid]['file_name'],
        'gt':        img_to_anns_int[iid]
    } for iid in ids]

train_data = build_sample(train_sample)
test_data  = build_sample(test_sample)

train_out = OUTPUT_DIR / '04_train_sample.json'
test_out  = OUTPUT_DIR / '04_test_sample.json'

with open(train_out, 'w') as f: json.dump(train_data, f, indent=2)
with open(test_out,  'w') as f: json.dump(test_data,  f, indent=2)

print(f'Train: {len(train_data)} images → {train_out}')
print(f'Test : {len(test_data)}  images → {test_out}')
print(f'\nExample GT entry:')
print(json.dumps(train_data[0], indent=2)[:300])

Train: 2000 images → ..\outputs\results\04_train_sample.json
Test : 1000  images → ..\outputs\results\04_test_sample.json

Example GT entry:
{
  "image_id": 240490,
  "file_name": "COCO_train2014_000000240490.jpg",
  "gt": [
    {
      "box": [
        221.9,
        291.4,
        33.0,
        39.9
      ],
      "text": "08"
    },
    {
      "box": [
        221.7,
        317.3,
        17.7,
        15.8
      ],
      "text": "0


## 4. Copy Sampled Images for Upload

In [4]:
PROJECT_ROOT = Path('..')

In [5]:
import shutil

# ── Update to your local train2014 folder ──
IMG_SRC_DIR = PROJECT_ROOT / 'data/raw/train2014'
IMG_OUT_DIR = PROJECT_ROOT / 'outputs/benchmark_images'
# ───────────────────────────────────────────

IMG_OUT_DIR.mkdir(parents=True, exist_ok=True)

all_samples  = train_data + test_data
missing      = 0
copied       = 0

for s in all_samples:
    src = IMG_SRC_DIR / s['file_name']
    dst = IMG_OUT_DIR / s['file_name']
    if dst.exists():
        copied += 1
        continue
    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing += 1
        print(f'Missing: {src}')

print(f'\nCopied : {copied} images → {IMG_OUT_DIR}')
print(f'Missing: {missing}')
print('\n→ Upload outputs/benchmark_images/ to Drive at:')
print('  data/raw/benchmark_images/')
print('→ Upload outputs/results/04_train_sample.json and 04_test_sample.json to:')
print('  outputs/results/')


Copied : 3000 images → ..\outputs\benchmark_images
Missing: 0

→ Upload outputs/benchmark_images/ to Drive at:
  data/raw/benchmark_images/
→ Upload outputs/results/04_train_sample.json and 04_test_sample.json to:
  outputs/results/
